# M2: 고정예산 N/V 상품반응 임베딩 (Dunnhumby, seed 42)

LightGCN ID 64차원에 N 반응 1차원과 V-가격 반응 1차원만 추가합니다. 사용자 CLV 총량 `q_C`를 N/V 구성비에 따라 `b_N+b_V=q_C`로 정확히 배분하고, 상품 N 반응은 기존 상품 ID 임베딩의 단일 투영으로, V 반응은 전체 가격 백분위의 방향이 고정된 단일 축으로 표현합니다. 동일 초기화 `rho=0`, 실제 CLV, 사용자 degree 구간 안에서 CLV를 순열한 대조군을 모두 100 epoch로 학습하며 최종 test와 holdout은 생성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
SOURCE_COMMIT = 'decbef53045ebb3b5bb51c5ad61d634d11ebaa34'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
if len(SOURCE_COMMIT) != 40:
    raise RuntimeError('검토된 소스 커밋을 SOURCE_COMMIT에 고정해야 합니다')
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch origin; else git clone {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} checkout {SOURCE_COMMIT}
%cd {REPO_DIR}


In [ ]:
import json
import torch
from lightgcn_clv_fixed_budget_nv_response import (
    configure_fixed_budget_nv_response_run,
    preflight_summary,
    run_fixed_budget_nv_response_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_fixed_budget_nv_response_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
result_df = run_fixed_budget_nv_response_screen(cfg)


In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: M1, rho=0, 실제 CLV, degree-matched shuffle, ID/N/V ablation')
display(result_df)
print('2) 대조군별 전체·CLV 구간 성과')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) rho=0 대비 CLV 구간별 Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 실제 CLV 대 degree-matched shuffle Top-10 변경')
display(pd.DataFrame(result_df.attrs['attribution_overlap']))
print('5) N/V별 실제 점수 영향력')
display(pd.DataFrame(result_df.attrs['score_diagnostics']))
print('6) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
